# 🎬 FOIA-BOT for Google Colab

**Free video case bundler for investigative journalism and true crime research**

This notebook helps you run FOIA-BOT in Google Colab.

## Quick Start:
1. Run **Setup** (Cell 1) - First time only
2. Run **Process Case** (Cell 3) with your YouTube URL
3. Download the bundle from Google Drive

---

## 📦 Step 1: Setup (Run Once)

This cell:
- Mounts Google Drive
- Installs dependencies
- Sets up the FOIA-BOT environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q yt-dlp requests beautifulsoup4 feedparser newspaper3k lxml_html_clean pandas matplotlib

# Clone or update FOIA-BOT repo
import os
from pathlib import Path

repo_dir = Path('/content/drive/MyDrive/foia_bot/repo')
repo_dir.mkdir(parents=True, exist_ok=True)

# Check if repo already exists
if not (repo_dir / 'video_ingest.py').exists():
    print("\n📥 Cloning FOIA-BOT repository...")
    !git clone https://github.com/jj55222/FOIA-BOT.git /tmp/foia_bot_temp
    !cp -r /tmp/foia_bot_temp/* {repo_dir}/
    !rm -rf /tmp/foia_bot_temp
else:
    print("\n✓ FOIA-BOT already installed")

# Change to repo directory
os.chdir(str(repo_dir))
print(f"\n✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")

## 🎯 Step 2: Configure Your Case

Edit the variables below with your YouTube URL and case name

In [ ]:
# ===== EDIT THESE VALUES =====
YOUTUBE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # Change this!
CASE_NAME = "my_case"  # Optional: custom name for your case
# ============================

## 🚀 Step 3: Process the Case

Run this to process the entire pipeline

In [ ]:
import subprocess
import os
import re
from pathlib import Path
from datetime import datetime

# Navigate to repo directory
repo_dir = Path('/content/drive/MyDrive/foia_bot/repo')
os.chdir(str(repo_dir))

# Generate case directory name
if CASE_NAME and CASE_NAME != "my_case":
    slug = CASE_NAME
else:
    # Extract video ID from URL
    match = re.search(r'v=([a-zA-Z0-9_-]+)', YOUTUBE_URL)
    slug = match.group(1) if match else f"case_{int(datetime.now().timestamp())}"

case_dir = f"case_{datetime.now().strftime('%Y%m%d')}_{slug}"

print(f"🎬 FOIA-BOT Case Processing")
print(f"={'='*60}")
print(f"Video URL: {YOUTUBE_URL}")
print(f"Case directory: {case_dir}")
print(f"={'='*60}\n")

# Create and enter case directory
case_path = repo_dir / case_dir
case_path.mkdir(exist_ok=True)
os.chdir(str(case_path))

print(f"Working in: {os.getcwd()}\n")

# Define pipeline steps
steps = [
    ("Video Ingest", f'python3 {repo_dir}/video_ingest.py "{YOUTUBE_URL}"'),
    ("Court Documents", f'python3 {repo_dir}/court_docs.py'),
    ("FOIA Media", f'python3 {repo_dir}/foia_media_scraper.py'),
    ("News Scraping", f'python3 {repo_dir}/news_scraper.py'),
    ("Packaging", f'python3 {repo_dir}/packager.py'),
]

# Run each step
for i, (step_name, command) in enumerate(steps, 1):
    print(f"\n[{i}/5] {step_name}")
    print("─" * 60)

    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True
        )

        # Print output
        if result.stdout:
            print(result.stdout)
        if result.stderr and result.returncode != 0:
            print("STDERR:", result.stderr)

        if result.returncode == 0:
            print(f"✓ {step_name} completed")
        else:
            print(f"⚠️  {step_name} failed (continuing anyway)")

    except Exception as e:
        print(f"❌ Error in {step_name}: {e}")
        print("Continuing to next step...")

print(f"\n{'='*60}")
print("✅ Processing complete!")
print(f"{'='*60}")
print(f"\n📦 Bundle location:")
print(f"   {case_path}/{case_dir}_bundle.zip")
print(f"\n💡 Find your files in Google Drive at:")
print(f"   MyDrive/foia_bot/repo/{case_dir}/")

## 🛠️ Optional: Run Individual Steps

If you want to run steps manually, use the cells below

In [ ]:
# Step 1: Video Ingest Only
!python3 /content/drive/MyDrive/foia_bot/repo/video_ingest.py "{YOUTUBE_URL}"

In [ ]:
# Step 2: Court Documents Only
!python3 /content/drive/MyDrive/foia_bot/repo/court_docs.py

In [ ]:
# Step 3: FOIA Media Only
!python3 /content/drive/MyDrive/foia_bot/repo/foia_media_scraper.py

In [ ]:
# Step 4: News Scraping Only
!python3 /content/drive/MyDrive/foia_bot/repo/news_scraper.py

In [ ]:
# Step 5: Package Bundle Only
!python3 /content/drive/MyDrive/foia_bot/repo/packager.py

## 📥 Download Your Bundle

Your processed case is saved in Google Drive. You can:
1. Access it directly in Google Drive at `MyDrive/foia_bot/repo/case_YYYYMMDD_name/`
2. Download the zip bundle using the cell below

In [ ]:
# Download the bundle to your local machine
from google.colab import files
import glob

# Find the most recent bundle
bundles = sorted(glob.glob('/content/drive/MyDrive/foia_bot/repo/case_*/*_bundle.zip'))
if bundles:
    latest_bundle = bundles[-1]
    print(f"Downloading: {latest_bundle}")
    files.download(latest_bundle)
else:
    print("❌ No bundles found. Make sure processing completed successfully.")